In [ ]:
!pip install transformers
!pip install emoji
!pip install soynlp

In [ ]:
%cd drive/MyDrive/국립국어원

/content/drive/MyDrive/국립국어원


In [ ]:
import json, os, sys, torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch.nn as nn

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    BertTokenizerFast,
    AlbertModel,
    AutoModel
)
from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader

def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list


In [ ]:
import emoji
# from soynlp.normalizer import repeat_normalize
import re

em = emoji.EMOJI_DATA.keys()
emojis = '|'.join(em)
repeatchars_pattern = re.compile('(\w|\W)\\1{2,}')
doublespace_pattern = re.compile('\s+')

def repeat_normalize(sent, num_repeats=2):
    if num_repeats > 0:
        sent = repeatchars_pattern.sub('\\1' * num_repeats, sent)
    sent = doublespace_pattern.sub(' ', sent)
    return sent.strip()

def remove_duplicate_words(input_string):
    words = input_string.split()

    unique_words = []
    for word in words:
        if word not in unique_words:
            unique_words.append(word)

    result_string = ' '.join(unique_words)
    return result_string

def text_preprocessing(text, order):
    # text = re.sub(f'[{emojis}]', '', text)
    for e in em:
        text = text.replace(e, '')
    if text != '':
        if text[0] == ' ':
            text = repeat_normalize(text, num_repeats=2)
            text = ' ' + text
        else:
            text = repeat_normalize(text, num_repeats=2)
            text = text + ' '
    text = text.replace('&others&', '')
    text = remove_duplicate_words(text)
    if order == 0:
        text = text.lstrip()
    elif order == 1:
        text = text.rstrip()
    else:
        text = repeat_normalize(text, num_repeats=2)
    return text

def data_preprocessing(text, target, target_idx):
    if target is None:
        text = text_preprocessing(text, 2)
        text = text.strip()
        return text, target, target_idx
    left_text = text_preprocessing(text[:target_idx[0]], 0)
    right_text = text_preprocessing(text[target_idx[1]:], 1)
    target_text = text_preprocessing(target, 2)
    new_text = left_text + target_text + right_text
    new_target_idx = [len(left_text), len(left_text)+len(target_text)]
    return new_text, target_text, new_target_idx

def json_preprocessing(json_data):
    for i, data in tqdm(enumerate(json_data)):
        text = data['input']['form']
        target = data['input']['target']['form']
        target_idx = [data['input']['target']['begin'], data['input']['target']['end']]
        new_text, new_target, new_target_idx = data_preprocessing(text, target, target_idx)

        json_data[i]['input']['form'] = new_text
        json_data[i]['input']['target']['form'] = new_target
        json_data[i]['input']['target']['begin'] = new_target_idx[0]
        json_data[i]['input']['target']['end'] = new_target_idx[1]
    return json_data

def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')

train_df = jsonlload('data/nikluge-ea-2023-train.jsonl')
valid_df = jsonlload('data/nikluge-ea-2023-dev.jsonl')
test_df = jsonlload('data/nikluge-ea-2023-test.jsonl')

for data_df, file_name in zip([train_df, valid_df, test_df], ['new_nikluge-ea-2023-train_2.jsonl', 'new_nikluge-ea-2023-dev_2.jsonl', 'new_nikluge-ea-2023-test_2.jsonl']):
    new_df = json_preprocessing(data_df)
    jsonldump(new_df, 'new_data/{}'.format(file_name))

In [ ]:
import json
import numpy as np
import pandas as pd

with open('data/data_unk_preprocessing_1.json', "r", encoding="utf-8") as f:
    data_unk = eval(f.read())

In [ ]:
{'왤케' : '왜이렇게',
'왤캐' : '왜이렇게',
'왤ㄹㄹ케' : '왜이렇게',
'왤까' : '왜일까',
'쳣' : '쳤',
'쟤' : '저 아이',
'피카쮸' : '피카츄',
'셋쮸' : '세츠',
'가좍' : '가족',
'갑좍' : '갑자기',
'아좍시' : '아저씨',
'줫' : '줬',
'좃' : '좆',
'횬' : '현',
'맠프' : '마크 프사',
'맠토' : '마크 스토',
'맠스' : '마크 인스',
'으맠' : '음악',
'맠걸릐' : '막걸리',
'맠쥐마' : '막지마',
'됏' : '됐',
'웟' : '웠',
'닼사' : '다크사이드',
'닼나' : '다크나이트',
'닼' : '다ㅋ',
'넼' : '네ㅋ',
'뎈' : '데ㅋ',
'곀' : '겨ㅋ',
'곸' : '고ㅋ',
'앜' : '아ㅋ',
'댘' : '대ㅋ',
'횽' : '형',
'컾흘' : '커플',
'컾' : '커플',
'븨앺' : '브이앱',
'븨' : '브이',
'뵤' : '보이',
'섡' : '선',
'쮝' : '찍',
'슼' : '슥',
'뀨' : '꾸',
'썅' : '상',
'닠' : '니ㅋ',
'쇼셏' : '쇼츠',
'앟' : '아ㅎ',
'쌰' : '샤',
'땨' : '다',
'븜' : '움',
'젛' : '좋',
'왹져' : '외계인',
'왹케' : '왜이렇게',
'낰낰' : '노크노크',
'낰' : '나ㅋ',
'햌' : '해ㅋ',
'즤' : '지',
'뢈' : '람',
'따땃' : '따뜻',
'땃뜨' : '타트',
'밯' : '발',
'⫬' : 'ㅋ',
'㉪' : 'ㅋ',
'𐌅' : 'ㅋ',
'ｦ' : 'ㅋ',
'뱌' : '바',
'챱' : '찹',
'햄쥑' : '햄스터',
'쥑쥑' : '쥐',
'튭' : '튜브',
'졔' : '제',
'놧' : '놨',
'뽝' : '빡',
'욬' : '요ㅋ',
'켯' : '켰',
'쩰' : '젤',
'팃팅' : '티켓팅',
'팃' : '티',
'됴아' : '좋아',
'라됴' : '라디오',
'비됴' : '비디오',
'됴' : '죠',
'귡' : '균',
'좔' : '잘',
'짴' : '짜ㅋ',
'뾱' : '뽁',
'뮨' : '문',
'얔' : '야ㅋ',
'삘' : '필',
'밈시뫈' : '임시완',
'힣' : '히히',
'진촤' : '진짜',
'촤령' : '촬영',
'촤' : '차',
'섴' : '서ㅋ',
'쒸' : '씨',
'앺' : '앱',
'먄남' : '만남',
'먄' : '면',
'떰' : '때',
'빉옵은' : '영빈오빠는',
'빉옵' : '영빈오빠',
'빉' : '영빈',
'줭' : '정',
'웤' : '워ㅋ',
'눙' : '눈',
'윷' : '유',
'싳' : '싶',
'칰' : '칙',
'뜀' : '뛰움',
'얜' : '얘는',
'뫄' : '모',
'짘' : '지ㅋ',
'썻' : '썼',
'잏는' : '있는',
'잏러' : '일러',
'잏케' : '이렇게',
'잏' : '이ㅎ',
'빻' : '빠',
'엏' : '어ㅎ',
'풉니' : '풀어줍니',
'풉시' : '풀어줍시',
'풉' : '푸하',
'듕' : '중',
'퉤' : '퉷',
'묜' : '면',
'웑' : '원',
'시펏' : '싶었'
'펏' : '펐',
'왐마' : '엄마',
'몀' : '면'
}

In [ ]:
for e in aa[57][1]:
    p = False
    for w in ['됴아', '라됴']:
        if w in e:
            p = True
            break
    if not p:
        print(e)

In [ ]:
list(set(['.ᐟ', 'ᐟ', 'ʚ', '⃫', 'ᰔ', '◡̈', '̫', 'ᙏ̤̫', '·', '̤', '̫', 'ᙏ', '·', '̑', '๑', '◡', '･', 'T', '°', '˂', '˃', '̣', '̥', 'ິ', 'ີ', '་', 'ི', 'ེ', 'ཻ', 'ོ', 'ཽ', 'ཾ', 'ྀ', '᷄', '᷅', '•', '⌓', '꒦', '̧', '་', 'ི', 'ེ', 'ཻ', 'ོ', 'ཽ', 'ཾ', 'ྀ', '⌓', '⍢', '⸝', 'Ĭ', 'ƪ', 'ɞ', 'ʚ', 'ˀ', 'ˏ', '̖', '̗', '̫', '̮', '̳͟͞͞', '͟͟͞͞', '་ེི', '༼', '༽', 'ི', 'ཻྀཽཾ', 'ཻྀཽཾ⌓', 'ᐕ', 'ᐟ', 'ᙏ', 'ᰔ', 'ᰔᩚ', 'ᴥ', '᷄', '᷄⌓', '᷅', '⃫', '⌄', '⌓', '⌓꒦', '⌔', '⍢', '◜', '◝', '⫬', 'Ⱉ', '⸌', '⸍ɞ', 'ꌩ', '꒦', 'ꖶ', 'ꫀꪜꪖ']))
['ེ', 'Ĭ', '̮', '⫬', '°', 'ꌩ', 'ཽ', 'ˏ', '.ᐟ', '̗', '᷄', '༼', '̫', '๑', '·', '･', 'ི', '̳͟͞͞', '˂', '͟͟͞͞', '་ེི', '⌄', 'ོ', 'ཻྀཽཾ', 'ྀ', '̖', 'ᰔ', '⍢', '◡', '̥', '⌓', '᷅', 'Ⱉ', 'ᴥ', '̣', 'ཻ', 'ᐕ', '་', '꒦', 'ᐟ', 'ɞ', '᷄⌓', '⌔', '༽', '̑', '⃫', 'ᰔᩚ', '⸝', 'ƪ', 'ʚ', 'ີ', 'ꫀꪜꪖ', 'ᙏ̤̫', '⌓꒦', '̤', '⸍ɞ', '̧', '•', 'ິ', '⸌', '◜', 'T', 'ꖶ', 'ᙏ', '˃', 'ˀ', 'ཻྀཽཾ⌓', 'ཾ', '◝', '◡̈']

In [ ]:
# aa = sorted(data_unk.items(), key=lambda x : len(x[1]), reverse=True)
aa[100]

('몀', ['햇으몀', '하몀', '잡채인장몀', '하몀', '풀렸으몀', '붕방붕방거리몀서', '헤엄쳤으몀', '충분했으몀'])

In [ ]:
len(aa)

742

In [ ]:
sorted(list(set(''.join(aa[91][1]))))

[')', '̧', '་', 'ི', 'ེ', 'ཻ', 'ོ', 'ཽ', 'ཾ', 'ྀ', '⌓', '⍢', '⸝']

In [ ]:
sorted(list(data_unk.keys()))

In [ ]:
tt = {'왤케' : '왜이렇게',
'왤캐' : '왜이렇게',
'왤ㄹㄹ케' : '왜이렇게',
'왤까' : '왜일까',
'쳣' : '쳤',
'쟤' : '저 아이',
'피카쮸' : '피카츄',
'셋쮸' : '세츠',
'가좍' : '가족',
'갑좍' : '갑자기',
'아좍시' : '아저씨',
'줫' : '줬',
'좃' : '좆',
'횬' : '현',
'맠프' : '마크 프사',
'맠토' : '마크 스토',
'맠스' : '마크 인스',
'으맠' : '음악',
'맠걸릐' : '막걸리',
'맠쥐마' : '막지마',
'됏' : '됐',
'웟' : '웠',
'닼사' : '다크사이드',
'닼나' : '다크나이트',
'닼' : '다ㅋ',
'넼' : '네ㅋ',
'뎈' : '데ㅋ',
'곀' : '겨ㅋ',
'곸' : '고ㅋ',
'앜' : '아ㅋ',
'댘' : '대ㅋ',
'횽' : '형',
'컾흘' : '커플',
'컾' : '커플',
'븨앺' : '브이앱',
'븨' : '브이',
'뵤' : '보이',
'섡' : '선',
'쮝' : '찍',
'슼' : '슥',
'뀨' : '꾸',
'썅' : '상',
'닠' : '니ㅋ',
'쇼셏' : '쇼츠',
'앟' : '아ㅎ',
'쌰' : '샤',
'땨' : '다',
'븜' : '움',
'젛' : '좋',
'왹져' : '외계인',
'왹케' : '왜이렇게',
'낰낰' : '노크노크',
'낰' : '나ㅋ',
'햌' : '해ㅋ',
'즤' : '지',
'뢈' : '람',
'따땃' : '따뜻',
'땃뜨' : '타트',
'밯' : '발',
'⫬' : 'ㅋ',
'㉪' : 'ㅋ',
'𐌅' : 'ㅋ',
'ｦ' : 'ㅋ',
'뱌' : '바',
'챱' : '찹',
'햄쥑' : '햄스터',
'쥑쥑' : '쥐',
'튭' : '튜브',
'졔' : '제',
'놧' : '놨',
'뽝' : '빡',
'욬' : '요ㅋ',
'켯' : '켰',
'쩰' : '젤',
'팃팅' : '티켓팅',
'팃' : '티',
'됴아' : '좋아',
'라됴' : '라디오',
'비됴' : '비디오',
'됴' : '죠',
'귡' : '균',
'좔' : '잘',
'짴' : '짜ㅋ',
'뾱' : '뽁',
'뮨' : '문',
'얔' : '야ㅋ',
'삘' : '필',
'밈시뫈' : '임시완',
'힣' : '히히',
'진촤' : '진짜',
'촤령' : '촬영',
'촤' : '차',
'섴' : '서ㅋ',
'쒸' : '씨',
'앺' : '앱',
'먄남' : '만남',
'먄' : '면',
'떰' : '때',
'빉옵은' : '영빈오빠는',
'빉옵' : '영빈오빠',
'빉' : '영빈',
'줭' : '정',
'웤' : '워ㅋ',
'눙' : '눈',
'윷' : '유',
'싳' : '싶',
'칰' : '칙',
'뜀' : '뛰움',
'얜' : '얘는',
'뫄' : '모',
'짘' : '지ㅋ',
'썻' : '썼',
'잏는' : '있는',
'잏러' : '일러',
'잏케' : '이렇게',
'잏' : '이ㅎ',
'빻' : '빠',
'엏' : '어ㅎ',
'풉니' : '풀어줍니',
'풉시' : '풀어줍시',
'풉' : '푸하',
'듕' : '중',
'퉤' : '퉷',
'묜' : '면',
'웑' : '원',
'시펏' : '싶었',
'펏' : '펐',
'왐마' : '엄마',
'몀' : '면'
}.items()
for t in tt:
    print(t, ',')

In [ ]:
tt = [('왤케', '왜이렇게') ,
('왤캐', '왜이렇게') ,
('왤ㄹㄹ케', '왜이렇게') ,
('왤까', '왜일까') ,
('쳣', '쳤') ,
('쟤', '저 아이') ,
('피카쮸', '피카츄') ,
('셋쮸', '세츠') ,
('가좍', '가족') ,
('갑좍', '갑자기') ,
('아좍시', '아저씨') ,
('줫', '줬') ,
('좃', '좆') ,
('횬', '현') ,
('맠프', '마크 프사') ,
('맠토', '마크 스토') ,
('맠스', '마크 인스') ,
('으맠', '음악') ,
('맠걸릐', '막걸리') ,
('맠쥐마', '막지마') ,
('됏', '됐') ,
('웟', '웠') ,
('닼사', '다크사이드') ,
('닼나', '다크나이트') ,
('닼', '다ㅋ') ,
('넼', '네ㅋ') ,
('뎈', '데ㅋ') ,
('곀', '겨ㅋ') ,
('곸', '고ㅋ') ,
('앜', '아ㅋ') ,
('댘', '대ㅋ') ,
('횽', '형') ,
('컾흘', '커플') ,
('컾', '커플') ,
('븨앺', '브이앱') ,
('븨', '브이') ,
('뵤', '보이') ,
('섡', '선') ,
('쮝', '찍') ,
('슼', '슥') ,
('뀨', '꾸') ,
('썅', '상') ,
('닠', '니ㅋ') ,
('쇼셏', '쇼츠') ,
('앟', '아ㅎ') ,
('쌰', '샤') ,
('땨', '다') ,
('븜', '움') ,
('젛', '좋') ,
('왹져', '외계인') ,
('왹케', '왜이렇게') ,
('낰낰', '노크노크') ,
('낰', '나ㅋ') ,
('햌', '해ㅋ') ,
('즤', '지') ,
('뢈', '람') ,
('따땃', '따뜻') ,
('땃뜨', '타트') ,
('밯', '발') ,
('⫬', 'ㅋ') ,
('㉪', 'ㅋ') ,
('𐌅', 'ㅋ') ,
('ｦ', 'ㅋ') ,
('뱌', '바') ,
('챱', '찹') ,
('햄쥑', '햄스터') ,
('쥑쥑', '쥐') ,
('튭', '튜브') ,
('졔', '제') ,
('놧', '놨') ,
('뽝', '빡') ,
('욬', '요ㅋ') ,
('켯', '켰') ,
('쩰', '젤') ,
('팃팅', '티켓팅') ,
('팃', '티') ,
('됴아', '좋아') ,
('라됴', '라디오') ,
('비됴', '비디오') ,
('됴', '죠') ,
('귡', '균') ,
('좔', '잘') ,
('짴', '짜ㅋ') ,
('뾱', '뽁') ,
('뮨', '문') ,
('얔', '야ㅋ') ,
('삘', '필') ,
('밈시뫈', '임시완') ,
('힣', '히히') ,
('진촤', '진짜') ,
('촤령', '촬영') ,
('촤', '차') ,
('섴', '서ㅋ') ,
('쒸', '씨') ,
('앺', '앱') ,
('먄남', '만남') ,
('먄', '면') ,
('떰', '때') ,
('빉옵은', '영빈오빠는') ,
('빉옵', '영빈오빠') ,
('빉', '영빈') ,
('줭', '정') ,
('웤', '워ㅋ') ,
('눙', '눈') ,
('윷', '유') ,
('싳', '싶') ,
('칰', '칙') ,
('뜀', '뛰움') ,
('얜', '얘는') ,
('뫄', '모') ,
('짘', '지ㅋ') ,
('썻', '썼') ,
('잏는', '있는') ,
('잏러', '일러') ,
('잏케', '이렇게') ,
('잏', '이ㅎ') ,
('빻', '빠') ,
('엏', '어ㅎ') ,
('풉니', '풀어줍니') ,
('풉시', '풀어줍시') ,
('풉', '푸하') ,
('듕', '중') ,
('퉤', '퉷') ,
('묜', '면') ,
('웑', '원') ,
('시펏', '싶었') ,
('펏', '펐') ,
('왐마', '엄마') ,
('몀', '면')]
with open('word_change_1.json', "w", encoding='utf-8') as f:
    f.write(json.dumps(tt, ensure_ascii=False))

In [ ]:
ss = ['ེ', 'Ĭ', '̮', '⫬', '°', 'ꌩ', 'ཽ', 'ˏ', '.ᐟ', '̗', '᷄', '༼', '̫', '๑', '·', '･', 'ི', '̳͟͞͞', '˂', '͟͟͞͞', '་ེི', '⌄', 'ོ', 'ཻྀཽཾ', 'ྀ', '̖', 'ᰔ', '⍢', '◡', '̥', '⌓', '᷅', 'Ⱉ', 'ᴥ', '̣', 'ཻ', 'ᐕ', '་', '꒦', 'ᐟ', 'ɞ', '᷄⌓', '⌔', '༽', '̑', '⃫', 'ᰔᩚ', '⸝', 'ƪ', 'ʚ', 'ີ', 'ꫀꪜꪖ', 'ᙏ̤̫', '⌓꒦', '̤', '⸍ɞ', '̧', '•', 'ິ', '⸌', '◜', 'T', 'ꖶ', 'ᙏ', '˃', 'ˀ', 'ཻྀཽཾ⌓', 'ཾ', '◝', '◡̈']
with open('word_del_1.json', "w", encoding='utf-8') as f:
    f.write(json.dumps(ss, ensure_ascii=False))